In [1]:
import pandas as pd
import torch
import tqdm

from agfn.gym import MAG_HITL, LogReward
from agfn.policies import ForwardPolicyHITL, BackwardPolicyHITL

from agfn.gflownet import GFlowNet

from agfn.commons import is_mag

torch.manual_seed(42)

In [2]:
device = "cpu"
assert device == "cpu", "Currently, only CPU is supported"

## Load data

In [3]:
data_path = "data/chain4_dat01.csv"
data = pd.read_csv(data_path)
data = torch.tensor(data.values, dtype=torch.get_default_dtype(), device=device)

num_nodes = data.shape[1]

In [4]:
hidden_dim = 128
num_layers = 2

pf = ForwardPolicyHITL(num_nodes, hidden_dim, num_layers, device=device)
pb = BackwardPolicyHITL(device=device)

gfn = GFlowNet(pf, pb, criterion="tb", device=device)

In [5]:
log_reward = LogReward(data, device=device)

initial_state = MAG_HITL(num_nodes, 256, log_reward)
log_reward.warmup(gfn, initial_state=initial_state, num_batches=8)

In [6]:
state = gfn.sample(
    initial_state
)

p, s = state.parents_and_spouses

is_graph_mag, th = is_mag(p, s, return_th=True)
assert is_graph_mag.all(), (th.mean(), th)

## Train an AGFN

In [7]:
opt = torch.optim.Adam([
    {"params": gfn.pf.parameters(), "lr": 1e-3},
    {"params": gfn.log_z, "lr": 1e-1},
])

num_epochs = 100

pbar = tqdm.tqdm(range(num_epochs))

for _ in pbar:
    initial_state = MAG_HITL(num_nodes, 8, log_reward)
    opt.zero_grad()
    loss = gfn(initial_state)
    loss.backward()
    # opt.step()

    pbar.set_postfix(loss=loss.item())

  0%|          | 0/100 [00:00<?, ?it/s]

tensor([], dtype=torch.int64) tensor([], size=(0, 6))
tensor([5]) tensor([[0., 1., 0., 0., 0., 0.]])


AssertionError: 